# 14. Pretraining Data and Objective

## Problem

模型在预训练阶段到底学了什么？为什么一个看起来很朴素的目标，也就是 next-token prediction，最后能支撑出语言能力、代码能力、知识共现和一部分推理能力？

## Dependencies

建议先理解 token、logits、softmax、cross entropy 和 context length 这些基础概念。


## What pretraining is really doing

很多人第一次看到预训练目标时，会觉得它“只是猜下一个 token”。这个理解太浅了。

预训练真正做的事情是：

1. 给模型一段前文。
2. 让模型对下一个 token 给出一个完整概率分布。
3. 如果正确 token 的概率不够高，就反向传播，调整参数。
4. 在海量数据、海量位置、海量上下文上重复这个过程。

这个目标虽然简单，但约束非常强。因为模型想把下一个 token 猜准，它就被迫去利用：

- 语法结构
- 长短程依赖
- 词语共现
- 事实模式
- 代码语法与 API 使用习惯
- 常见推理模板

所以预训练不是在“背答案”，而是在反复压缩文本世界里的统计结构。

从整个训练链路看，pretraining 的职责是先做一个足够强的 base model。它不会直接把模型变成一个好助手，但它决定了后续 SFT、reward model、RLHF 能不能站在稳固底座上继续做行为塑形。


## Core math

单个位置上的 next-token loss 可以写成：

$$
L_t = -\log p_\theta(x_t \mid x_{<t})
$$

这里：

- $x_{<t}$ 表示位置 $t$ 之前的所有 token。
- $x_t$ 表示当前位置真正应该出现的 token。
- $p_\theta(x_t \mid x_{<t})$ 表示模型在参数 $\theta$ 下给正确 token 分配的概率。

如果把一整段序列都算进去，常见写法就是：

$$
L_{seq} = -\sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

或者取平均：

$$
L_{seq}^{mean} = -\frac{1}{T} \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

对整个训练集而言，本质上就是把无数个 token 位置上的这个损失不断累积。

这条公式的含义非常直接：**如果模型给正确 token 的概率越低，损失就越大；如果模型越来越愿意把概率压到正确 token 上，损失就越小。**


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 这是一个单位置预测示例。
# 词表大小为 5，模型输出 5 个 logit，表示它对 5 个 token 的原始打分。
logits = np.array([1.2, 0.3, 2.4, -0.5, 0.1])
target_id = 2


def softmax(x):
    # 先减最大值，避免指数溢出。这是 softmax 的常见数值稳定写法。
    shifted = x - np.max(x)
    exp = np.exp(shifted)
    return exp / np.sum(exp)


probs = softmax(logits)
loss = -np.log(probs[target_id])

print('logits =', logits)
print('probabilities =', probs)
print('probability assigned to the correct token =', probs[target_id])
print('cross entropy loss =', loss)


## From text to teacher forcing targets

预训练不是把整段文本一次性作为一个整体标签去学习，而是把它切成很多“看前文，猜下一位”的局部任务。

例如一句很短的 token 序列：

- 输入第 1 步：`[BPE]`，目标是预测后面的 token
- 输入第 2 步：`[BPE, is]`，目标是预测再后面的 token
- 输入第 3 步：`[BPE, is, useful]`，继续预测下一个

这就是 teacher forcing 的核心：训练时我们把真实前文喂给模型，而不是让模型每一步都吃自己刚生成的输出。这样训练更稳定，也更容易并行。


In [ ]:
# 这里用一个玩具 token 序列展示 teacher forcing 的“错位一格”结构。
# 输入序列是前文，目标序列是向左平移一位后的真实答案。

token_ids = np.array([11, 25, 7, 42, 9])
inputs = token_ids[:-1]
targets = token_ids[1:]

print('full token sequence =', token_ids)
print('model inputs        =', inputs)
print('teacher targets     =', targets)

for step, (prefix_end, target) in enumerate(zip(range(1, len(token_ids)), targets), start=1):
    prefix = token_ids[:prefix_end]
    print(f'step {step}: context {prefix.tolist()} -> predict {int(target)}')


In [ ]:
# 再看一个最小 batch 版本。这里每一行都表示一个位置上的词表 logits。
# 真实训练里，shape 往往是 [batch_size, seq_len, vocab_size]。

batch_logits = np.array([
    [2.0, 0.5, 0.2],
    [0.1, 1.5, 0.3],
    [0.2, 0.4, 1.8],
])
batch_targets = np.array([0, 1, 2])

# 对每一行 logits 做 softmax，得到每个位置的 token 概率分布。
shifted = batch_logits - np.max(batch_logits, axis=1, keepdims=True)
batch_probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)

# 只取每个位置上“正确 token”对应的概率，再取负对数。
per_token_loss = -np.log(batch_probs[np.arange(len(batch_targets)), batch_targets])
mean_loss = per_token_loss.mean()

print('batch probabilities =')
print(batch_probs)
print('per-token loss =', per_token_loss)
print('mean loss =', mean_loss)


## Context length, token budget, and why scale matters

预训练时，还有三个工程量非常重要：

- **batch size**：一次并行处理多少条样本
- **context length**：每条样本最多看多长前文
- **training tokens**：整个训练阶段一共消费多少 token

它们不是一个东西。

- context length 决定模型单次能看到多长依赖。
- batch size 决定每一步并行吞吐。
- training tokens 更接近“总训练量”。

一个模型即使结构不变，训练 token 数翻很多倍，能力也可能明显变化。因为它看到的语言模式更多，参数被更新得更充分。


In [ ]:
# 这个例子只是帮助建立量级直觉，不是工业训练配置。

batch_size = 8
context_length = 1024
steps = 5000

tokens_per_step = batch_size * context_length
total_training_tokens = tokens_per_step * steps

print('batch_size =', batch_size)
print('context_length =', context_length)
print('tokens processed per step =', tokens_per_step)
print('total tokens after all steps =', total_training_tokens)

# 观察把 context length 翻倍后，总 token 吞吐如何变化。
longer_context = 2048
longer_tokens_per_step = batch_size * longer_context
print('tokens per step with longer context =', longer_tokens_per_step)


## Why this simple objective is enough to build a base model

最容易误解的一点是：next-token prediction 看起来太简单，为什么会有这么强的结果？

答案不是“目标神奇”，而是因为约束足够密、数据足够大、参数足够多。

为了在海量上下文里把下一个 token 猜得更准，模型会被迫学习：

- 哪些词会一起出现
- 哪些语法结构更常见
- 哪些代码模式会接着哪些调用
- 哪些问题后面常跟哪些回答框架
- 哪些知识片段经常共同出现

也就是说，预训练不是直接教模型“如何当一个助手”，而是先让它形成一个足够强的语言和知识压缩器。

## Common mistakes

- 把预训练理解成背诵固定答案。更准确地说，它是在拟合条件分布。
- 只盯住 loss 数值，而不理解 loss 本质上是“正确 token 的对数概率惩罚”。
- 把 batch size、context length、training tokens 混成一个概念。它们都影响成本，但作用不同。
- 以为预训练结束就已经得到最终产品。实际上它更像是在浇地基。

## Summary

预训练阶段最核心的事，就是通过 next-token prediction 把广义语言分布压进参数。它决定模型是否有足够强的底层表示能力。后面的 SFT、reward model、PPO、GRPO，更多是在这个底座之上继续塑形，而不是替代这个底座。
